In [4]:
#!/usr/bin/env python3
"""
HAWKEYE — SYNTHETIC DATA GENERATOR
====================================
Generates a synthetic banking dataset that preserves the *signals* the
proven AML model relies on, at the same class imbalance (~0.8% mules).

The mule-vs-legit distinction is encoded via the same statistical signatures
that gave us AUC=0.998 on the real RBI NFPC data:

  Pass-through behaviour      → low |balance_after_txn| after credits
  Structuring                  → amounts clustered at 45-49K (just below 50K)
  Round-amount preference      → spikes at 1K/5K/10K/25K/50K
  Off-hours activity           → night/weekend bias
  Fan-out asymmetry            → many credit-counterparties, few debit ones (or vice versa)
  Counterparty hub-sharing     → mules cluster around the same counterparties
  Burstiness                   → activity concentrated in 1-2 months out of 6
  Shared IP addresses          → mules share IPs with other mules
  Recent KYC / mobile change   → fresh contact info before activity spike
  New-account-high-volume      → recently-opened accounts moving large sums

Outputs (saved to /kaggle/working/hawkeye_synthetic/ or ./hawkeye_synthetic/):
  ├── synthetic_accounts.parquet        # static features per account/employee
  ├── synthetic_transactions.parquet    # event stream (one row per event)
  ├── synthetic_labels.parquet          # ground truth (is_mule == is_insider_threat)
  ├── synthetic_events.jsonl            # Kafka-replay-ready newline-delimited JSON
  └── synthesis_metadata.json           # config used + summary stats

The HAWKEYE backend re-labels these for insider-threat semantics:
  account_id     → employee_id      (e.g. ACC_0001234 → EMP_0001234)
  counterparty   → system/resource  (e.g. CP_8392 → SYS_TREASURY_42)
  amount         → records_accessed (or sensitivity score)
  txn_type C/D   → READ / WRITE
  ip_address     → workstation_ip   (kept as-is)
  is_mule        → is_insider_threat
"""
import numpy as np, pandas as pd, json, os, gc
from datetime import datetime, timedelta
import argparse

# ──────────────────────────────────────────────────────────────────────────
# CONFIG — tune for demo size / realism trade-off
# ──────────────────────────────────────────────────────────────────────────
DEFAULT_CFG = dict(
    n_accounts          = 10_000,    # total accounts/employees
    mule_rate           = 0.0113,     # 0.8% — matches real RBI dataset
    n_counterparties    = 3_000,     # total unique counterparties (systems)
    n_mule_hub_cps      = 50,        # counterparties shared mostly by mules
    n_mule_ips          = 80,        # IPs shared by mule cohort
    n_legit_ips         = 5_000,     # IP pool for legit accounts
    months              = 6,         # observation window
    seed                = 42,
    output_dir          = './hawkeye_synthetic',
)

# ──────────────────────────────────────────────────────────────────────────
def gen_static_features(rng, n, mule_mask):
    """Account static features mirroring real-data distributions."""
    today = pd.Timestamp('2026-05-05')

    # account age — mules slightly skew younger
    age_d_legit = rng.gamma(shape=4.0, scale=600, size=n)
    age_d_mule  = rng.gamma(shape=2.0, scale=300, size=n)
    age_d = np.where(mule_mask, age_d_mule, age_d_legit).clip(15, 7300)

    # KYC days — mules sometimes overdue
    kyc_d_legit = rng.uniform(0, 365, n)
    kyc_d_mule  = rng.uniform(0, 730, n)
    kyc_d = np.where(mule_mask, kyc_d_mule, kyc_d_legit)

    # Mobile change — mules dramatically more likely (recently changed)
    mob_change_recent = np.where(
        mule_mask,
        rng.binomial(1, 0.55, n),
        rng.binomial(1, 0.04, n)).astype(np.int8)
    mob_change_days = np.where(mob_change_recent == 1,
                                rng.uniform(1, 365, n),
                                rng.uniform(366, 3000, n))

    # Average balance — mules low, legit lognormal
    avg_balance_legit = np.exp(rng.normal(10.5, 1.4, n))   # ~₹36K median
    avg_balance_mule  = np.exp(rng.normal(7.5, 1.0, n))    # ~₹1.8K median
    avg_balance = np.where(mule_mask, avg_balance_mule, avg_balance_legit)

    # Other static
    product_code = rng.choice(['SA01','SA02','SA03','CA01','CA02'], n,
                                p=[0.45, 0.20, 0.10, 0.15, 0.10])
    pf  = rng.choice([0,1,2], n, p=[0.6, 0.3, 0.1]).astype(np.int8)
    rur = rng.binomial(1, 0.32, n).astype(np.int8)
    nri = rng.binomial(1, 0.04, n).astype(np.int8)
    male = rng.binomial(1, 0.62, n).astype(np.int8)
    age = rng.normal(38, 12, n).clip(18, 85)

    # Digital footprint — mules tend to have minimal digital footprint
    n_digital_legit = rng.binomial(7, 0.55, n)
    n_digital_mule  = rng.binomial(7, 0.25, n)
    ndig = np.where(mule_mask, n_digital_mule, n_digital_legit).astype(np.int8)

    # PMJDY / scheme indicator — mules disproportionately PMJDY
    pmjdy = np.where(mule_mask,
                      rng.binomial(1, 0.45, n),
                      rng.binomial(1, 0.18, n)).astype(np.int8)

    return pd.DataFrame({
        'account_id':       [f'ACC_{i:08d}' for i in range(n)],
        'is_mule':          mule_mask.astype(np.int8),
        'product_code':     product_code,
        'product_family':   pf,
        'avg_balance':      avg_balance.astype(np.float32),
        'rural_branch':     rur,
        'nri_flag':         nri,
        'male':             male,
        'age':              age.astype(np.float32),
        'age_d':            age_d.astype(np.float32),
        'kyc_d':            kyc_d.astype(np.float32),
        'mob_change_recent':mob_change_recent,
        'mob_change_days':  mob_change_days.astype(np.float32),
        'ndig':             ndig,
        'pmjdy':            pmjdy,
        'account_opening_date': [(today - pd.Timedelta(days=int(d))).date().isoformat()
                                  for d in age_d],
    })

# ──────────────────────────────────────────────────────────────────────────
def gen_transactions(rng, accts_df, cfg):
    """Event-stream generation. Each account generates 6 months of transactions
       with mule-like or legit-like statistical signatures."""
    n_cp        = cfg['n_counterparties']
    n_hub       = cfg['n_mule_hub_cps']
    n_mule_ip   = cfg['n_mule_ips']
    n_legit_ip  = cfg['n_legit_ips']
    months      = cfg['months']

    cps        = np.array([f'CP_{i:06d}' for i in range(n_cp)])
    mule_hubs  = cps[:n_hub]                   # first N reserved for mule sharing
    legit_cps  = cps[n_hub:]
    mule_ips   = np.array([f'10.0.{i//256}.{i%256}' for i in range(n_mule_ip)])
    legit_ips  = np.array([f'192.168.{i//256}.{i%256}' for i in range(n_legit_ip)])

    end_date   = pd.Timestamp('2026-05-01')
    start_date = end_date - pd.Timedelta(days=30*months)

    rows, events = [], []
    txn_id = 0

    for idx, row in accts_df.iterrows():
        is_mule = bool(row['is_mule'])
        aid     = row['account_id']

        # ─── Volume profile ─────────────────────────────────────────────
        if is_mule:
            # Burstiness: 60% of mules concentrate activity in 1-2 months
            n_txn = int(rng.gamma(shape=3.0, scale=80))   # mean ~240
            burst = rng.random() < 0.6
        else:
            n_txn = int(rng.gamma(shape=2.0, scale=25))   # mean ~50
            burst = False
        n_txn = max(1, min(n_txn, 2000))

        # ─── Timestamps ─────────────────────────────────────────────────
        if burst:
            burst_month = rng.integers(0, months)
            burst_start = start_date + pd.Timedelta(days=30*burst_month)
            burst_end   = burst_start + pd.Timedelta(days=rng.integers(20, 50))
            ts_secs = rng.uniform(burst_start.timestamp(),
                                  min(burst_end, end_date).timestamp(), n_txn)
        else:
            ts_secs = rng.uniform(start_date.timestamp(), end_date.timestamp(), n_txn)
        ts = pd.to_datetime(ts_secs, unit='s')

        # ─── Hour bias ──────────────────────────────────────────────────
        if is_mule and rng.random() < 0.45:
            # 45% of mules bias toward night (23-05) and weekends
            night_idx = rng.random(n_txn) < 0.55
            night_hours = rng.choice(np.r_[23, np.arange(0, 6)], night_idx.sum())
            ts = pd.Series(ts)
            for i, h in zip(np.where(night_idx)[0], night_hours):
                ts.iloc[i] = ts.iloc[i].replace(hour=int(h))
            ts = pd.DatetimeIndex(ts)

        # ─── Amounts ────────────────────────────────────────────────────
        if is_mule:
            mode = rng.choice(['structuring','round','passthrough','mixed'],
                                p=[0.30, 0.25, 0.30, 0.15])
            if mode == 'structuring':
                amts = rng.uniform(45_000, 49_999, n_txn)
            elif mode == 'round':
                amts = rng.choice([1_000, 5_000, 10_000, 25_000, 50_000], n_txn,
                                    p=[0.10, 0.20, 0.30, 0.25, 0.15])
            elif mode == 'passthrough':
                amts = np.exp(rng.normal(9.2, 0.6, n_txn))   # tightly grouped
                amts = amts.clip(2_000, 80_000)
            else:
                amts = np.exp(rng.normal(8.5, 1.2, n_txn))
            # add occasional XLG transactions
            xlg_mask = rng.random(n_txn) < 0.05
            amts[xlg_mask] = rng.uniform(100_000, 500_000, xlg_mask.sum())
        else:
            amts = np.exp(rng.normal(7.0, 1.5, n_txn))   # broad distribution
            amts = amts.clip(10, 500_000)

        # ─── Txn type C/D ───────────────────────────────────────────────
        if is_mule:
            # Mules are typically transit accounts: ~50/50 with high asymmetry
            tt = rng.choice(['C','D'], n_txn, p=[0.52, 0.48])
        else:
            tt = rng.choice(['C','D'], n_txn, p=[0.40, 0.60])  # debit-heavy normal

        # ─── Counterparties ─────────────────────────────────────────────
        if is_mule:
            # 70% of credits flow through hub counterparties
            n_uniq_cp_c = max(2, int(rng.gamma(2.0, 8)))    # high fan-in
            n_uniq_cp_d = max(1, int(rng.gamma(1.5, 2)))    # low fan-out
            cp_pool_c = rng.choice(mule_hubs, min(n_uniq_cp_c, n_hub), replace=False)
            cp_pool_d = rng.choice(legit_cps, n_uniq_cp_d, replace=False)
        else:
            n_uniq_cp_c = max(1, int(rng.gamma(1.5, 3)))
            n_uniq_cp_d = max(1, int(rng.gamma(2.0, 5)))
            cp_pool_c = rng.choice(legit_cps, n_uniq_cp_c, replace=False)
            cp_pool_d = rng.choice(legit_cps, n_uniq_cp_d, replace=False)
        cps_assigned = np.where(tt=='C',
                                  rng.choice(cp_pool_c, n_txn),
                                  rng.choice(cp_pool_d, n_txn))

        # ─── IP addresses ───────────────────────────────────────────────
        if is_mule and rng.random() < 0.7:
            n_ips = max(1, int(rng.gamma(1.2, 1.5)))   # few unique IPs
            ip_pool = rng.choice(mule_ips, min(n_ips, n_mule_ip), replace=False)
        else:
            n_ips = max(1, int(rng.gamma(2.0, 2.0)))
            ip_pool = rng.choice(legit_ips, n_ips, replace=False)
        ips = rng.choice(ip_pool, n_txn)

        # ─── Balance after txn ──────────────────────────────────────────
        if is_mule:
            # Pass-through: balance stays near zero
            bal = np.abs(rng.normal(0, 800, n_txn))
        else:
            bal = np.abs(rng.normal(row['avg_balance'], row['avg_balance']*0.3, n_txn))
            bal = bal.clip(0, None)

        # ─── MCC + channel ──────────────────────────────────────────────
        mcc = rng.choice(['6011','5411','5814','6051','4900','5912','7372'], n_txn,
                          p=[0.30, 0.18, 0.12, 0.10, 0.10, 0.10, 0.10])
        channel = rng.choice(['UPI','NEFT','RTGS','IMPS','ATM','POS'], n_txn,
                              p=[0.45, 0.18, 0.07, 0.15, 0.10, 0.05])

        # ─── Build rows ─────────────────────────────────────────────────
        for i in range(n_txn):
            t = ts[i]
            row_dict = {
                'transaction_id':         f'TXN_{txn_id:010d}',
                'account_id':             aid,
                'transaction_timestamp':  t.isoformat(),
                'amount':                 float(amts[i]) * (1 if tt[i]=='C' else -1),
                'txn_type':               tt[i],
                'counterparty_id':        cps_assigned[i],
                'mcc_code':               mcc[i],
                'channel':                channel[i],
                'ip_address':             ips[i],
                'balance_after_transaction': float(bal[i]),
            }
            rows.append(row_dict)
            # Insider-threat overlay (HAWKEYE narrative format)
            events.append({
                **row_dict,
                'employee_id':           aid.replace('ACC_','EMP_'),
                'system_resource':       cps_assigned[i].replace('CP_','SYS_'),
                'access_type':           'READ' if tt[i]=='C' else 'WRITE',
                'records_accessed':      int(amts[i] / 100),
                'workstation_ip':        ips[i],
                'is_after_hours':        bool(t.hour < 9 or t.hour >= 18),
                'is_weekend':            bool(t.dayofweek >= 5),
            })
            txn_id += 1

        if (idx+1) % 500 == 0:
            print(f"    {idx+1}/{len(accts_df):,} accounts | {txn_id:,} txns")

    return pd.DataFrame(rows), events

# ──────────────────────────────────────────────────────────────────────────
def main(cfg):
    rng = np.random.default_rng(cfg['seed'])
    os.makedirs(cfg['output_dir'], exist_ok=True)

    n     = cfg['n_accounts']
    n_mu  = max(1, int(cfg['mule_rate'] * n))
    print(f"Generating {n:,} accounts ({n_mu:,} mules = {n_mu/n:.3%})")

    mule_mask                 = np.zeros(n, dtype=bool)
    mule_mask[rng.choice(n, n_mu, replace=False)] = True

    print("[1/3] Static account features...")
    accts_df = gen_static_features(rng, n, mule_mask)

    print("[2/3] Transactions + insider-threat events...")
    txn_df, events = gen_transactions(rng, accts_df, cfg)

    print("[3/3] Saving artifacts...")
    out = cfg['output_dir']

    # accounts
    accts_df.to_parquet(f'{out}/synthetic_accounts.parquet', index=False)
    # labels (separate, for clean train/serve split)
    accts_df[['account_id','is_mule']].rename(
        columns={'is_mule':'is_insider_threat'}
    ).assign(is_mule=accts_df['is_mule']).to_parquet(
        f'{out}/synthetic_labels.parquet', index=False)
    # transactions
    txn_df.to_parquet(f'{out}/synthetic_transactions.parquet', index=False)
    # JSONL stream (Kafka replay format)
    with open(f'{out}/synthetic_events.jsonl', 'w') as f:
        for ev in sorted(events, key=lambda e: e['transaction_timestamp']):
            f.write(json.dumps(ev) + '\n')

    # metadata
    meta = {
        'config':                 cfg,
        'n_accounts':             n,
        'n_mules':                int(n_mu),
        'mule_rate':              float(n_mu/n),
        'n_transactions':         len(txn_df),
        'avg_txns_per_acct':      float(len(txn_df)/n),
        'total_volume_inr':       float(txn_df['amount'].abs().sum()),
        'date_range_start':       txn_df['transaction_timestamp'].min(),
        'date_range_end':         txn_df['transaction_timestamp'].max(),
        'unique_counterparties':  int(txn_df['counterparty_id'].nunique()),
        'unique_ips':             int(txn_df['ip_address'].nunique()),
        'generated_at':           datetime.now().isoformat(),
    }
    with open(f'{out}/synthesis_metadata.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"\nDone. Saved to {out}/")
    for fn in sorted(os.listdir(out)):
        sz = os.path.getsize(f'{out}/{fn}') / 1024 / 1024
        print(f"  {fn:<35} {sz:>8.2f} MB")
    print(f"\nMule rate: {n_mu/n:.3%}  |  Total events: {len(txn_df):,}")
    print(f"Use synthetic_events.jsonl as Kafka replay source for HAWKEYE demo.")

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--n_accounts',     type=int,   default=DEFAULT_CFG['n_accounts'])
    parser.add_argument('--mule_rate',      type=float, default=DEFAULT_CFG['mule_rate'])
    parser.add_argument('--months',         type=int,   default=DEFAULT_CFG['months'])
    parser.add_argument('--output_dir',     type=str,   default=DEFAULT_CFG['output_dir'])
    parser.add_argument('--seed',           type=int,   default=DEFAULT_CFG['seed'])
    # parse_known_args so this works inside Jupyter/Kaggle/Colab cells
    # (which inject their own kernel flags like -f /tmp/kernel-xxx.json)
    args, _unknown = parser.parse_known_args()

    cfg = {**DEFAULT_CFG, **vars(args)}
    main(cfg)

Generating 10,000 accounts (112 mules = 1.120%)
[1/3] Static account features...
[2/3] Transactions + insider-threat events...
    500/10,000 accounts | 24,704 txns
    1000/10,000 accounts | 51,698 txns
    1500/10,000 accounts | 77,107 txns
    2000/10,000 accounts | 104,180 txns
    2500/10,000 accounts | 129,511 txns
    3000/10,000 accounts | 155,995 txns
    3500/10,000 accounts | 180,629 txns
    4000/10,000 accounts | 206,071 txns
    4500/10,000 accounts | 232,408 txns
    5000/10,000 accounts | 258,719 txns
    5500/10,000 accounts | 283,594 txns
    6000/10,000 accounts | 308,304 txns
    6500/10,000 accounts | 334,569 txns
    7000/10,000 accounts | 363,148 txns
    7500/10,000 accounts | 388,064 txns
    8000/10,000 accounts | 416,069 txns
    8500/10,000 accounts | 442,224 txns
    9000/10,000 accounts | 467,126 txns
    9500/10,000 accounts | 492,406 txns
    10000/10,000 accounts | 516,650 txns
[3/3] Saving artifacts...

Done. Saved to ./hawkeye_synthetic/
  synthesis_m